# Install Dependecies

In [1]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [48]:
import ipynbname
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import time
from collections import Counter
from datasets import load_dataset
from pathlib import Path
from tqdm.std import tqdm
from transformers import AutoTokenizer


# Set `ROOT_DIR`

In [4]:
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

/home/tlvj/msc_datalogi/2_semester/nlp/msc-nlp-2026/project_notebooks


# Import datasets

In [ ]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

### Tokeniser

In [6]:
xlm_tokeniser = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 2 Week 36: Data and Rule-Based Baselines
Download the dataset and inspect its columns. Report Item 1 separately by
language and split, plus overall example counts and answerability proportions.
Compute Item 2 from the training questions separately by language. Apply
Item 3 to every answerable example in both splits.

3. verify programmatically that every answerable item’s answer equals the
substring beginning at answer start, and report the number checked and
any failures.

Implement and evaluate two answerability baselines: (i) the majority-class
baseline estimated from the training split and (ii) a deterministic rule-based
classifier that uses only the question and context. The rule may use tokenisation,
lexical features or machine translation, but no labelled validation examples or
trained answerability/QA model. Discuss what information the rule can and
cannot exploit in this cross-lingual setting.

## Week 36.1
Report the number of examples, answerable/unanswerable proportions, median and interquartile range of tokenised question and context lengths (a table is sufficient; plots are optional), missing values and exact duplicate question–context pairs.

In [7]:
def token_len(texts):
    return [len(xlm_tokeniser(t)["input_ids"]) for t in texts]

In [8]:
df_train["q_len"] = token_len(df_train["question"])
df_train["c_len"] = token_len(df_train["context"])
df_val["q_len"] = token_len(df_val["question"])
df_val["c_len"] = token_len(df_val["context"])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (650 > 512). Running this sequence through the model will result in indexing errors


### Compute the number of answerable / unanswerable / pct_unanswerable

In [47]:
# train
counts = df_train.groupby(["lang"])["answerable"].agg(n="count", answerable="sum")
counts["unanswerable"] = counts["n"] - counts["answerable"]
counts["pct_unanswerable"] = (100 * counts["unanswerable"] / counts["n"]).round(0)
print(counts.to_markdown())

| lang   |    n |   answerable |   unanswerable |   pct_unanswerable |
|:-------|-----:|-------------:|---------------:|-------------------:|
| ar     | 2558 |         2303 |            255 |                 10 |
| bn     | 2598 |         2449 |            149 |                  6 |
| fi     | 2126 |         1863 |            263 |                 12 |
| ja     | 2301 |         1928 |            373 |                 16 |
| ko     | 2422 |         2359 |             63 |                  3 |
| ru     | 1983 |         1750 |            233 |                 12 |
| te     | 1355 |         1310 |             45 |                  3 |


In [44]:
# val
counts = df_val.groupby("lang")["answerable"].agg(n="count", answerable="sum")
counts["unanswerable"] = counts["n"] - counts["answerable"]
counts["pct_unanswerable"] = (100 * counts["unanswerable"] / counts["n"]).round(0)
print(counts.to_markdown())

| lang   |   n |   answerable |   unanswerable |   pct_unanswerable |
|:-------|----:|-------------:|---------------:|-------------------:|
| ar     | 415 |          363 |             52 |                 13 |
| bn     | 476 |          371 |            105 |                 22 |
| fi     | 528 |          380 |            148 |                 28 |
| ja     | 456 |          287 |            169 |                 37 |
| ko     | 356 |          337 |             19 |                  5 |
| ru     | 396 |          284 |            112 |                 28 |
| te     | 384 |          291 |             93 |                 24 |


### Overall counts

In [34]:
# train
n = len(df_train)
answerable = df_train["answerable"].sum()
print("n:", n, "| answerable:", answerable, "| unanswerable:", n - answerable,
      "| pct_unanswerable:", round(100 * (n - answerable) / n, 2))

n: 15343 | answerable: 13962 | unanswerable: 1381 | pct_unanswerable: 9.0


In [35]:
# val
n = len(df_val)
answerable = df_val["answerable"].sum()
print("n:", n, "| answerable:", answerable, "| unanswerable:", n - answerable,
      "| pct_unanswerable:", round(100 * (n - answerable) / n, 2))

n: 3011 | answerable: 2313 | unanswerable: 698 | pct_unanswerable: 23.18


### Median and IQR of question / context length

In [41]:
# train
g = df_train.groupby("lang")
lengths = pd.DataFrame({
    "q_median": g["q_len"].median(),
    "q_q25": g["q_len"].quantile(0.25),
    "q_q75": g["q_len"].quantile(0.75),
    "c_median": g["c_len"].median(),
    "c_q25": g["c_len"].quantile(0.25),
    "c_q75": g["c_len"].quantile(0.75),
})
lengths["q_iqr"] = lengths["q_q75"] - lengths["q_q25"]
lengths["c_iqr"] = lengths["c_q75"] - lengths["c_q25"]
print(lengths.to_markdown())

| lang   |   q_median |   q_q25 |   q_q75 |   c_median |   c_q25 |   c_q75 |   q_iqr |   c_iqr |
|:-------|-----------:|--------:|--------:|-----------:|--------:|--------:|--------:|--------:|
| ar     |         13 |      11 |      15 |        133 |    89   |   191   |       4 |     102 |
| bn     |         17 |      14 |      21 |        130 |    87   |   186   |       7 |      99 |
| fi     |         12 |      10 |      14 |        132 |    90   |   191   |       4 |     101 |
| ja     |         14 |      12 |      17 |        132 |    88   |   188   |       5 |     100 |
| ko     |         14 |      12 |      16 |        127 |    85   |   181   |       4 |      96 |
| ru     |         15 |      12 |      18 |        137 |    88.5 |   194.5 |       6 |     106 |
| te     |         13 |      11 |      16 |        121 |    80   |   171   |       5 |      91 |


In [42]:
# val
g = df_val.groupby("lang")
lengths = pd.DataFrame({
    "q_median": g["q_len"].median(),
    "q_q25": g["q_len"].quantile(0.25),
    "q_q75": g["q_len"].quantile(0.75),
    "c_median": g["c_len"].median(),
    "c_q25": g["c_len"].quantile(0.25),
    "c_q75": g["c_len"].quantile(0.75),
})
lengths["q_iqr"] = lengths["q_q75"] - lengths["q_q25"]
lengths["c_iqr"] = lengths["c_q75"] - lengths["c_q25"]
print(lengths.to_markdown())

| lang   |   q_median |   q_q25 |   q_q75 |   c_median |   c_q25 |   c_q75 |   q_iqr |   c_iqr |
|:-------|-----------:|--------:|--------:|-----------:|--------:|--------:|--------:|--------:|
| ar     |         12 |      11 |      15 |      131   |   89.5  |  190.5  |       4 |  101    |
| bn     |         17 |      15 |      21 |      140   |   90    |  180.25 |       6 |   90.25 |
| fi     |         12 |      11 |      15 |      149   |  104    |  207    |       4 |  103    |
| ja     |         14 |      12 |      17 |      120   |   93.75 |  166    |       5 |   72.25 |
| ko     |         14 |      12 |      16 |      123.5 |   80.75 |  183.5  |       4 |  102.75 |
| ru     |         14 |      12 |      17 |      145.5 |   91.75 |  191    |       5 |   99.25 |
| te     |         14 |      12 |      18 |      154   |  114    |  208    |       6 |   94    |


### Missing values

In [39]:
print(df_train.isna().sum().to_markdown())
print("-------------------------")
print(df_val.isna().sum().to_markdown())

|               |     0 |
|:--------------|------:|
| question      |     0 |
| context       |     0 |
| lang          |     0 |
| answerable    |     0 |
| answer_start  |     0 |
| answer        |     0 |
| answer_inlang | 15093 |
| q_len         |     0 |
| c_len         |     0 |
-------------------------
|               |    0 |
|:--------------|-----:|
| question      |    0 |
| context       |    0 |
| lang          |    0 |
| answerable    |    0 |
| answer_start  |    0 |
| answer        |    0 |
| answer_inlang | 2511 |
| q_len         |    0 |
| c_len         |    0 |


### Duplicate pairs

In [40]:
print("train:", df_train.duplicated(subset=["question", "context"]).sum())
print("val:", df_val.duplicated(subset=["question", "context"]).sum())

train: 17
val: 0


## Week 36.2
Report the five most common question tokens and their counts for each language, together with an English translation, and explain your tokenisation.